# Cross-Entropy and NLL Loss

**Goal:** Implement NLL loss and softmax-cross-entropy from scratch in PyTorch.
Assert each against PyTorch's own `F.nll_loss(F.log_softmax(...))` and `F.cross_entropy`.
Show that the combined softmax+CE gradient is `(p − one_hot(y)) / N` and validate it against autograd.
Cover class-index targets and briefly discuss `ignore_index` and reduction modes.

See also: `[[softmax]]`, `[[mle-and-nll]]`

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## Stable logsumexp

`logsumexp(z) = log(sum_j exp(z_j))` is numerically unstable when logits are large.
The stable version subtracts the maximum before exponentiating:

```
logsumexp(z) = m + log(sum_j exp(z_j - m))   where m = max(z)
```

This keeps all arguments to `exp` non-positive, avoiding overflow.

In [2]:
# Use cpu + float64 for precise manual checks
cpu64 = torch.device("cpu")
dtype64 = torch.float64

torch.manual_seed(0)
B, C = 128, 5

logits = torch.randn(B, C, dtype=dtype64, device=cpu64)
targets = torch.randint(0, C, (B,), device=cpu64)


def stable_logsumexp(z: torch.Tensor) -> torch.Tensor:
    """Numerically stable logsumexp over last dimension. Shape: (B,)"""
    m = z.max(dim=-1, keepdim=True).values           # (B, 1)
    return m.squeeze(-1) + torch.log(torch.exp(z - m).sum(dim=-1))


def manual_log_softmax(z: torch.Tensor) -> torch.Tensor:
    """log(softmax(z)) = z - logsumexp(z). Shape same as z."""
    return z - stable_logsumexp(z).unsqueeze(-1)      # (B, C)


lse_manual = stable_logsumexp(logits)
lse_torch  = torch.logsumexp(logits, dim=-1)

print(f"logsumexp max |err|: {(lse_manual - lse_torch).abs().max().item():.2e}")
assert torch.allclose(lse_manual, lse_torch, atol=1e-10), "logsumexp mismatch"
print("logsumexp assertion passed.")

# Test on extreme logits (would overflow with naive exp)
extreme = torch.tensor([[1e38, 0.0, 0.0]], dtype=dtype64)
lse_extreme = stable_logsumexp(extreme)
print(f"\nExtreme logit logsumexp (stable): {lse_extreme.item():.4f}")
print(f"Expected ~1e38 (dominated by max): {lse_extreme.item() > 1e37}")


logsumexp max |err|: 0.00e+00
logsumexp assertion passed.

Extreme logit logsumexp (stable): 99999999999999997748809823456034029568.0000
Expected ~1e38 (dominated by max): True


## log_softmax Validation

`log_softmax(z)_i = z_i − logsumexp(z)` is more numerically stable than computing `log(softmax(z))` separately, because `softmax` may produce tiny values that lose precision under `log`.

In [3]:
log_probs_manual = manual_log_softmax(logits)          # (B, C)
log_probs_torch  = F.log_softmax(logits, dim=-1)        # (B, C)

print(f"log_softmax max |err|: {(log_probs_manual - log_probs_torch).abs().max().item():.2e}")
assert torch.allclose(log_probs_manual, log_probs_torch, atol=1e-10), "log_softmax mismatch"
print("log_softmax assertion passed.")

# Demonstrate instability of naive log(softmax(z)) on extreme logits
extreme2 = torch.tensor([[1e38, 0.0, 0.0]], dtype=dtype64)
naive = torch.log(torch.softmax(extreme2, dim=-1))
stable = manual_log_softmax(extreme2)
print(f"\nNaive  log(softmax) for extreme: {naive}")
print(f"Stable log_softmax  for extreme: {stable}")
print("Naive version produces -inf due to overflow in softmax denominator.")


log_softmax max |err|: 8.88e-16
log_softmax assertion passed.

Naive  log(softmax) for extreme: tensor([[0., -inf, -inf]], dtype=torch.float64)
Stable log_softmax  for extreme: tensor([[ 0.0000e+00, -1.0000e+38, -1.0000e+38]], dtype=torch.float64)
Naive version produces -inf due to overflow in softmax denominator.


## NLL Loss from Log-Probabilities

Negative log likelihood for integer class targets:

```
NLL = -(1/N) * sum_b log_probs[b, y_b]
```

When the model already outputs log-probabilities (e.g., after `log_softmax`), use `F.nll_loss`.
This is equivalent to `F.cross_entropy(logits, targets)` when logits are provided raw.

In [4]:
def manual_nll_loss(log_probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """NLL loss: -(1/N) sum_b log_probs[b, y_b]. Reduction='mean'."""
    B = log_probs.shape[0]
    # Index the log-probability of the correct class for each sample
    correct_log_probs = log_probs[torch.arange(B), targets]   # (B,)
    return -correct_log_probs.mean()


nll_manual = manual_nll_loss(log_probs_manual, targets)
nll_torch  = F.nll_loss(log_probs_torch, targets)              # expects log-probs

print(f"NLL manual : {nll_manual.item():.8f}")
print(f"NLL torch  : {nll_torch.item():.8f}")
print(f"Max |err|  : {abs(nll_manual.item() - nll_torch.item()):.2e}")

assert torch.allclose(nll_manual, nll_torch, atol=1e-10), "NLL mismatch"
print("NLL assertion passed.")


NLL manual : 1.96987265
NLL torch  : 1.96987265
Max |err|  : 4.44e-16
NLL assertion passed.


## Cross-Entropy from Logits

`F.cross_entropy` fuses `log_softmax` + `nll_loss` in one stable operation.
Starting from logits `z` with correct class `c`:

```
L = -z_c + logsumexp(z)
```

This avoids materialising softmax probabilities and is always preferred over `log(softmax(z))`.

In [5]:
def manual_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Stable cross-entropy from logits. Reduction='mean'."""
    lse = stable_logsumexp(logits)                           # (B,)
    log_p_correct = logits[torch.arange(len(targets)), targets] - lse  # (B,)
    return -log_p_correct.mean()


ce_manual = manual_cross_entropy(logits, targets)
ce_torch   = F.cross_entropy(logits, targets)
ce_via_nll = F.nll_loss(F.log_softmax(logits, dim=-1), targets)

print(f"CE manual          : {ce_manual.item():.8f}")
print(f"F.cross_entropy    : {ce_torch.item():.8f}")
print(f"nll_loss+log_softmax: {ce_via_nll.item():.8f}")
print(f"Max |err| (manual vs F.cross_entropy): {(ce_manual - ce_torch).abs().item():.2e}")

assert torch.allclose(ce_manual, ce_torch, atol=1e-10), "CE manual vs F.cross_entropy mismatch"
assert torch.allclose(ce_manual, ce_via_nll, atol=1e-10), "CE manual vs nll+log_softmax mismatch"
print("Both cross-entropy assertions passed.")


CE manual          : 1.96987265
F.cross_entropy    : 1.96987265
nll_loss+log_softmax: 1.96987265
Max |err| (manual vs F.cross_entropy): 4.44e-16
Both cross-entropy assertions passed.


## Gradient: `(p − one_hot(y)) / N`

For a batch of N examples, the gradient of the mean cross-entropy with respect to logits `z` is:

```
dL/dz_i = softmax(z)_i - y_i      (per-example, where y_i = 1 for the correct class)
```

Averaged over the batch:
```
dL/dz = (softmax(z) - one_hot(y)) / N        shape: (N, C)
```

**Derivation (one example):**
```
L = -z_c + log(sum_j exp(z_j))
dL/dz_i = -1[i=c] + exp(z_i) / sum_j exp(z_j)
         = p_i - y_i
```

The softmax and cross-entropy gradients cancel to give this clean, efficient expression.

In [6]:
def manual_ce_gradient(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Gradient of mean cross-entropy w.r.t. logits: (softmax(z) - one_hot(y)) / N."""
    N = logits.shape[0]
    # Numerically stable softmax
    z_shift = logits - logits.max(dim=-1, keepdim=True).values
    exp_z = torch.exp(z_shift)
    probs = exp_z / exp_z.sum(dim=-1, keepdim=True)          # (N, C)

    # Subtract 1 at the correct class index (immutable: clone first)
    grad = probs.clone()
    grad[torch.arange(N), targets] -= 1.0
    return grad / N                                           # (N, C)


# --- Autograd reference ---
logits_ag = logits.clone().requires_grad_(True)
loss_ag = F.cross_entropy(logits_ag, targets)
loss_ag.backward()
autograd_grad = logits_ag.grad                               # (B, C)

manual_grad = manual_ce_gradient(logits, targets)

print(f"Gradient shape: {manual_grad.shape}")
print(f"Max |err| (manual grad vs autograd): {(manual_grad - autograd_grad).abs().max().item():.2e}")

assert torch.allclose(manual_grad, autograd_grad, atol=1e-10), "CE gradient mismatch vs autograd"
print("CE gradient assertion passed: (softmax(z) - one_hot(y)) / N matches autograd exactly.")


Gradient shape: torch.Size([128, 5])
Max |err| (manual grad vs autograd): 8.67e-19
CE gradient assertion passed: (softmax(z) - one_hot(y)) / N matches autograd exactly.


## Reduction Modes: `mean` vs `sum`

The `reduction` argument controls how per-example losses are combined:

- `mean` (default): divides by batch size — gradient scale is independent of batch size.
- `sum`: adds all losses — gradient scale grows linearly with batch size.

The choice affects the effective learning rate: switching from `mean` to `sum` multiplies all gradients by the batch size.

In [7]:
ce_mean = F.cross_entropy(logits, targets, reduction="mean")
ce_sum  = F.cross_entropy(logits, targets, reduction="sum")

print(f"Reduction='mean' : {ce_mean.item():.6f}")
print(f"Reduction='sum'  : {ce_sum.item():.6f}")
print(f"Ratio sum/mean   : {ce_sum.item() / ce_mean.item():.1f}  (expected ≈ {B})")

assert abs(ce_sum.item() / ce_mean.item() - B) < 0.01, "sum/mean ratio != batch size"
print(f"Assertion passed: sum = mean * B ({B})")

# Effect on gradient scale
logits_m = logits.clone().requires_grad_(True)
logits_s = logits.clone().requires_grad_(True)
F.cross_entropy(logits_m, targets, reduction="mean").backward()
F.cross_entropy(logits_s, targets, reduction="sum").backward()

ratio = logits_s.grad.abs().mean() / logits_m.grad.abs().mean()
print(f"\nGradient scale ratio (sum vs mean): {ratio.item():.2f}  (expected ≈ {B})")


Reduction='mean' : 1.969873
Reduction='sum'  : 252.143700
Ratio sum/mean   : 128.0  (expected ≈ 128)
Assertion passed: sum = mean * B (128)

Gradient scale ratio (sum vs mean): 128.00  (expected ≈ 128)


## `ignore_index` and Class Weights

**`ignore_index`** tells the loss to skip examples whose target equals a sentinel value (e.g., padding tokens in sequence models). Those examples contribute 0 to the loss and their gradients are zeroed.

**Class weights** (`weight` argument) re-scale each class's contribution to the loss. Useful for imbalanced datasets where rare classes would otherwise be overwhelmed.

Both modify gradient scale, so they interact with learning rate tuning.

In [8]:
# ignore_index demo: mark 10 examples as padding (label = -100)
targets_padded = targets.clone()
targets_padded[:10] = -100   # convention: -100 is the default ignore_index

ce_ignore = F.cross_entropy(logits, targets_padded, ignore_index=-100)
ce_no_ignore = F.cross_entropy(logits[10:], targets[10:])   # manually skip first 10

print(f"CE with ignore_index (skip 10): {ce_ignore.item():.6f}")
print(f"CE computed on remaining 118  : {ce_no_ignore.item():.6f}")
print("Note: F.cross_entropy with ignore_index computes mean over non-ignored examples.")

# Brief check: they should be close (not identical because reduction='mean' denominators differ)
print(f"Close? {torch.allclose(ce_ignore, ce_no_ignore, atol=1e-4)}")


CE with ignore_index (skip 10): 1.913597
CE computed on remaining 118  : 1.913597
Note: F.cross_entropy with ignore_index computes mean over non-ignored examples.
Close? True


## Idiomatic PyTorch

Always pass **raw logits** to `F.cross_entropy`. Do NOT apply softmax before calling it — the fused operation is numerically stable and avoids the `log(0)` problem.

In [9]:
import torch.nn as nn

torch.manual_seed(0)

# Simple linear classifier on configured device
classifier = nn.Linear(10, C).to(device=device)
X_dev = torch.randn(32, 10, device=device)
y_dev = torch.randint(0, C, (32,), device=device)

# Standard pattern: logits -> F.cross_entropy (no explicit softmax)
logits_dev = classifier(X_dev)
loss_dev = F.cross_entropy(logits_dev, y_dev)
loss_dev.backward()

print(f"F.cross_entropy loss on {device}: {loss_dev.item():.4f}")
print(f"Gradient norm (weight): {classifier.weight.grad.norm().item():.4f}")

# Show the equivalence chain for reference
log_p = F.log_softmax(logits_dev.detach(), dim=-1)
nll   = F.nll_loss(log_p, y_dev)
print(f"\nEquivalent F.nll_loss(F.log_softmax(...)): {nll.item():.4f}")
print(f"Matches F.cross_entropy: {torch.allclose(loss_dev.detach(), nll, atol=1e-5)}")


F.cross_entropy loss on mps: 1.7732
Gradient norm (weight): 0.6125

Equivalent F.nll_loss(F.log_softmax(...)): 1.7732
Matches F.cross_entropy: True


## Takeaways

- **NLL loss** = `−mean(log_probs[b, y_b])`. Use when the model already produces log-probabilities.
- **Cross-entropy from logits** = `−z_c + logsumexp(z)`. Always prefer this over `log(softmax(z))` — fused is numerically stable.
- **Gradient** of softmax cross-entropy w.r.t. logits is `(softmax(z) − one_hot(y)) / N`. This clean form arises because the softmax Jacobian and cross-entropy gradient cancel analytically.
- **Reduction matters**: `mean` gives a gradient scale independent of batch size; `sum` multiplies gradients by N. Changing reduction changes effective learning rate.
- **`ignore_index`** masks padded or missing targets — critical for sequence models with variable-length inputs.
- **Never softmax before cross_entropy**: `F.cross_entropy` expects raw logits. Applying softmax first introduces numerical error and a subtle distributional mismatch.
- See also: `[[softmax]]`, `[[mle-and-nll]]`, `[[label-smoothing]]`
